# Pediatric EEG V2 — full-cohort experiment

All 23 unique individuals; frozen 15/4/4 split. This notebook is an orchestration layer over the repository, not an alternate implementation. Run in a GPU runtime. Source EEG stays on the runtime disk; checkpoints and reports persist in your Drive. No test-based model selection.

Research/educational prototype. Not for diagnosis or clinical decisions.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")


In [ ]:
import os, subprocess, shutil
from pathlib import Path
REPO = "https://github.com/iampenuel/pediatric-eeg-seizure-v2.git"
PROJECT = Path("/content/pediatric-eeg-seizure-v2")
if not PROJECT.exists():
    subprocess.run(["git", "clone", REPO, str(PROJECT)], check=True)
os.chdir(PROJECT)
print(subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip())
subprocess.run(["nvidia-smi"], check=True)
print("Free runtime GiB:", round(shutil.disk_usage("/content").free / 2**30, 1))


In [ ]:
import sys
subprocess.run([sys.executable, "-m", "pip", "install", "--no-cache-dir", "-e", ".[data,train,export,serve,test]"], check=True)


In [ ]:
subprocess.run([sys.executable, "-m", "pytest", "-q"], check=True)


## Full-cohort run

The workflow prepares all eligible recordings, audits all 23 individuals, trains both models, freezes validation-only selection, evaluates both on held-out patients, and creates the demo bundle. Restart this cell after a runtime interruption: source downloads/cache are verified, training resumes from the previous completed epoch, and frozen runs are not retrained. Do not change the configuration after inspecting test outputs. Runtime GPU availability and duration are not guaranteed.

In [ ]:
DATA = "/content/chbmit-v2"
OUTPUT = "/content/drive/MyDrive/pediatric-eeg-seizure-v2/full-seed42"
subprocess.run([sys.executable, "scripts/run_study.py", "--data", DATA, "--output", OUTPUT], check=True)


In [ ]:
import json
summary = json.loads((Path(OUTPUT) / "reports/summary.json").read_text())
for name, report in summary.items():
    print(name, "pooled:", report["pooled"], "patient macro:", report["patient_macro"])
print("Demo release:", Path(OUTPUT) / "demo.tar.gz")
